# Cell 0 — Interactive backend

In [1]:
# If plots are static, install once in the env:
# pip install -U ipympl ipywidgets jupyterlab-widgets
%matplotlib widget

# Cell 1 — imports, I/O helpers, 2D FFT + circular notch mask

In [2]:
# === Cell-1: minimal helpers; GPU/CPU FFT; I/O ===
import os
import re
import json
import math
import warnings

import numpy as np
import imageio.v3 as iio
import matplotlib.pyplot as plt
import ipywidgets as W
from glob import glob
from IPython.display import display, clear_output

warnings.filterwarnings("ignore", category=UserWarning)

# ---- GPU if available ----
try:
    import cupy as cp
    _GPU = True
except Exception:
    import numpy as cp  # transparent CPU fallback
    _GPU = False

# ---- I/O helpers ----
def _natkey(s):
    return [int(t) if t.isdigit() else t.lower() for t in re.findall(r'\d+|\D+', s)]

def list_files_gray(folder, exts=("*.tif","*.tiff","*.png","*.jpg","*.jpeg","*.bmp")):
    files = sum([glob(os.path.join(folder, e)) for e in exts], [])
    files.sort(key=_natkey)
    assert files, f"No images in {folder}"
    return files

def read_gray(p):
    im = iio.imread(p)
    if im.ndim == 3:
        im = (0.2126*im[...,0] + 0.7152*im[...,1] + 0.0722*im[...,2]).astype(im.dtype)
    return im

def to_float01(img):
    if img.dtype == np.uint8:
        return img.astype(np.float32)/255.0, {"dtype": np.uint8, "scale": 255.0}
    if img.dtype == np.uint16:
        return img.astype(np.float32)/65535.0, {"dtype": np.uint16, "scale": 65535.0}
    if np.issubdtype(img.dtype, np.floating):
        return img.astype(np.float32), {"dtype": img.dtype, "scale": 1.0}
    return img.astype(np.float32), {"dtype": img.dtype, "scale": 1.0}

def from_float01(arr, meta):
    odt, scale = meta["dtype"], meta["scale"]
    if odt == np.uint8:
        return (np.clip(arr,0,1)*scale + 0.5).astype(np.uint8)
    if odt == np.uint16:
        return (np.clip(arr,0,1)*scale + 0.5).astype(np.uint16)
    if np.issubdtype(odt, np.floating):
        return arr.astype(odt)
    return arr

# ---- 2D frequency grid (cycles/pixel; Nyquist=0.5) ----
def kgrid2(shape):
    H, W = shape
    ky = cp.fft.fftfreq(H)  # 1D
    kx = cp.fft.fftfreq(W)  # 1D
    KY, KX = cp.meshgrid(ky, kx, indexing='ij')
    KR = cp.sqrt(KX**2 + KY**2)
    return KX, KY, KR

# ---- FFT helpers ----
def fft2(x):   return cp.fft.fft2(cp.asarray(x, dtype=cp.float32))
def ifft2(X):  return cp.fft.ifft2(X).real
def fftmag(F): return cp.log1p(cp.abs(cp.fft.fftshift(F)))  # log magnitude (shifted)

# ---- circular notch (±k mirrored) ----
def notch_passmask(shape, centers, default_radius):
    """
    Returns pass mask H (1=pass, 0=stop). centers: list of (ky, kx, r).
    Each notch removes a disk at (±ky, ±kx).
    """
    KX, KY, KR = kgrid2(shape)
    H = cp.ones(KR.shape, dtype=cp.float32)
    for (ky0, kx0, r) in centers:
        r = float(r if r is not None else default_radius)
        R1 = cp.sqrt((KX-kx0)**2 + (KY-ky0)**2)
        R2 = cp.sqrt((KX+kx0)**2 + (KY+ky0)**2)  # conjugate
        H *= (R1 > r).astype(cp.float32)
        H *= (R2 > r).astype(cp.float32)
    return H

def overlay_mask_on_mag(mag, H):
    # brighten pass (H=1), darken stop (H=0)
    M  = mag / (mag.max() + 1e-12)
    Hs = cp.fft.fftshift(H)
    vis = cp.clip(M*(0.6 + 0.4*Hs), 0, 1)
    return cp.asnumpy(vis)

def imgcoord_to_normfreq(xpix, ypix, W, H):
    # map clicked pixel (in shifted display) to normalized freq [-0.5,0.5]
    cx, cy = (W-1)/2.0, (H-1)/2.0
    kx = (xpix - cx) / W
    ky = (ypix - cy) / H
    return float(kx), float(ky)

# Cell 2 — minimal interactive notch UI (click-to-add, radius slider, undo/clear, save/load)

In [3]:
# === Cell-2 (TRIPLE-VIEW): original | FFT | filtered, with zoom/pan on ALL, STOP/PASS disks, depth & feather ===
# Uses helpers from your Cell-1 (read_gray, to_float01, fft2, ifft2, fftmag, list_files_gray, cp)

import os, json
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import display, clear_output

# ---------------- convenience ----------------
def to_np(x):  # cupy->numpy or pass-through
    return x.get() if hasattr(x, "get") else np.asarray(x)

def idx_grids_shifted(shape):
    """Pixel index grids for shifted FFT image (DC at image center)."""
    H, W = shape
    yi = cp.arange(H) - (H // 2)
    xi = cp.arange(W) - (W // 2)
    YI, XI = cp.meshgrid(yi, xi, indexing='ij')
    return XI, YI  # pixel units

def imgcoord_to_normfreq(xpix, ypix, W, H):
    cx, cy = (W-1)/2.0, (H-1)/2.0
    kx = (xpix - cx) / W
    ky = (ypix - cy) / H
    return float(kx), float(ky)

def apply_axis_zoom(ax, center_pix, zoom_factor, H, W):
    cx, cy = center_pix
    half_w = (W-1) / (2.0 * max(zoom_factor, 1e-6))
    half_h = (H-1) / (2.0 * max(zoom_factor, 1e-6))
    cx = float(np.clip(cx, half_w, (W-1) - half_w))
    cy = float(np.clip(cy, half_h, (H-1) - half_h))
    ax.set_xlim(cx - half_w, cx + half_w)
    ax.set_ylim(cy + half_h, cy - half_h)
    return (cx, cy)

# ---- Soft/hard disk in FFT PIXELS (r>=1). Returns 1 inside, 0 outside with cosine feather (SHIFTED coords) ----
def smooth_disk_px_shifted(shape, center_cyc, r_px, feather_px):
    H, W = shape
    XI, YI = idx_grids_shifted(shape)
    kx0, ky0 = center_cyc
    x0i = kx0 * W
    y0i = ky0 * H
    Rpix = cp.sqrt((XI - x0i)**2 + (YI - y0i)**2)
    r_px = max(1, int(r_px))
    f = max(0, int(feather_px))
    if f <= 0:
        return (Rpix <= r_px).astype(cp.float32)
    t = cp.clip((Rpix - r_px) / float(f), 0.0, 1.0)
    return (0.5 * (1.0 + cp.cos(np.pi * t))).astype(cp.float32)  # 1 at center → 0 outside ring

# ---- Build masks in SHIFTED coords, return UNSHIFTED for filtering ----
def build_masks(shape):
    """Return (Hs, Hp, H) in UNshifted coords. Hs: passmask after STOP, Hp: PASS union, H = Hs*Hp."""
    HsS = cp.ones(shape, dtype=cp.float32)                       # STOP passmask (shifted)
    HpS = cp.zeros(shape, dtype=cp.float32) if passes else cp.ones(shape, dtype=cp.float32)  # PASS (shifted)

    # STOP disks: attenuate by 'depth' inside disk (±k mirrored if enabled)
    for ky, kx, rpx in stops:
        Sd = smooth_disk_px_shifted(shape, (kx, ky), rpx, feather_px.value)
        if mirror.value:
            Sd = cp.maximum(Sd, smooth_disk_px_shifted(shape, (-kx, -ky), rpx, feather_px.value))
        HsS *= (1.0 - float(depth.value) * Sd)                   # partial-to-hard stop

    # PASS disks: union of disks (±k mirrored). If none, HpS=1 (pass-all)
    if passes:
        P = cp.zeros(shape, dtype=cp.float32)
        for ky, kx, rpx in passes:
            Pd = smooth_disk_px_shifted(shape, (kx, ky), rpx, feather_px.value)
            if mirror.value:
                Pd = cp.maximum(Pd, smooth_disk_px_shifted(shape, (-kx, -ky), rpx, feather_px.value))
            P = cp.maximum(P, Pd)
        HpS = P

    HS = HsS * HpS
    return cp.fft.ifftshift(HsS), cp.fft.ifftshift(HpS), cp.fft.ifftshift(HS)

# ================= USER CONFIG =================
try:
    DATA_DIR
except NameError:
    DATA_DIR = "/home/askiran/data/Peri_1"  # <-- set if not defined in another cell

files = list_files_gray(DATA_DIR)
z_mid  = len(files)//2

# ---------- STATE ----------
stops, passes, action_hist = [], [], []
_zoom_centers = {'orig': None, 'fft': None, 'filt': None}
_dragging = False
_drag_key = None
_drag_start = (0.0, 0.0)
_click_cids = []  # store mpl connection ids to disconnect on redraw

# ---------- WIDGETS ----------
z_slider  = W.IntSlider(value=z_mid, min=0, max=len(files)-1, step=1, description='z')
radius_px = W.IntSlider(value=4, min=1, max=128, step=1, description='radius(px)')
feather_px= W.IntSlider(value=3, min=0, max=64, step=1, description='feather(px)')
depth     = W.FloatSlider(value=1.0, min=0.1, max=1.0, step=0.05, readout_format='.2f', description='depth')
mask_alpha= W.FloatSlider(value=0.45, min=0.1, max=0.95, step=0.05, description='mask α')
zoom      = W.FloatLogSlider(value=1.0, base=2, min=0, max=4, step=0.1, description='Zoom ×')  # 1..16×
mode      = W.ToggleButtons(options=[("Add STOP","stop"),("Add PASS","pass"),
                                     ("Pan","pan"),("Set center","center")],
                            value="stop", description="Click")
mirror    = W.Checkbox(value=True, description="Mirror ±k")
btn_reset = W.Button(description="Reset view")
btn_undo  = W.Button(description="Undo last")
btn_clear = W.Button(description="Clear all", button_style='warning')
preset    = W.Text(value=os.path.join(DATA_DIR, "_mask2d_px.json"), description="Preset", layout=W.Layout(width="60%"))
btn_save  = W.Button(description="Save")
btn_load  = W.Button(description="Load")
log_out   = W.Output()
plot_out  = W.Output()

# ---------- DRAW / EVENTS ----------
def update_preview(*_):
    global _click_cids, _dragging, _drag_key
    with plot_out:
        clear_output(wait=True)

        # data
        s = to_float01(read_gray(files[z_slider.value]))[0]
        Hh, Ww = s.shape
        if _zoom_centers['orig'] is None:
            mid = ((Ww-1)/2.0, (Hh-1)/2.0)
            _zoom_centers['orig'] = mid
            _zoom_centers['fft']  = mid
            _zoom_centers['filt'] = mid

        F = fft2(s)
        Hs, Hp, H = build_masks(s.shape)      # UNshifted masks for filtering
        Sf = ifft2(F * H); Sf = to_np(Sf).astype(np.float32)

        mag = fftmag(F); vis = to_np(mag / (mag.max()+1e-12))  # normalized log-mag for display
        Hs_shift = to_np(cp.fft.fftshift(Hs))
        Hp_shift = to_np(cp.fft.fftshift(Hp))
        H_shift  = to_np(cp.fft.fftshift(H))

        # figure
        fig = plt.figure(figsize=(15,5.4))

        # Left: ORIGINAL slice
        axL = plt.subplot(1,3,1)
        p_lo, p_hi = np.percentile(s, (1,99))
        imL = axL.imshow(s, cmap="gray", vmin=p_lo, vmax=p_hi)
        axL.set_title("Original (spatial)"); axL.axis("off")

        # Middle: FFT (with overlays)
        axM = plt.subplot(1,3,2)
        axM.imshow(vis, cmap="gray")
        masked = 1.0 - H_shift
        if masked.max() > 0:
            axM.imshow(masked, cmap="Reds", alpha=float(mask_alpha.value), vmin=0, vmax=1)
        try: axM.contour(Hp_shift, levels=[0.5], colors='lime', linewidths=1)
        except Exception: pass
        thr = 1.0 - max(0.05, 0.5*float(depth.value))
        try: axM.contour(H_shift, levels=[thr], colors='yellow', linewidths=1)
        except Exception: pass
        axM.set_title("FFT (log) + mask overlays"); axM.axis("off")

        # Right: FILTERED slice
        axR = plt.subplot(1,3,3)
        imR = axR.imshow(Sf, cmap="gray", vmin=p_lo, vmax=p_hi)
        axR.set_title("Filtered (spatial)"); axR.axis("off")

        # Apply zoom to all three panes
        _zoom_centers['orig'] = apply_axis_zoom(axL, _zoom_centers['orig'], zoom.value, Hh, Ww)
        _zoom_centers['fft']  = apply_axis_zoom(axM, _zoom_centers['fft'],  zoom.value, Hh, Ww)
        _zoom_centers['filt'] = apply_axis_zoom(axR, _zoom_centers['filt'], zoom.value, Hh, Ww)

        # Bind events (per-axis behavior)
        for cid in _click_cids:
            try: fig.canvas.mpl_disconnect(cid)
            except: pass
        _click_cids = []

        ax2key = {axL: 'orig', axM: 'fft', axR: 'filt'}

        def on_press(ev):
            global _dragging, _drag_key, _drag_start
            if ev.inaxes not in ax2key or ev.xdata is None or ev.ydata is None:
                return
            key = ax2key[ev.inaxes]
            # Right click or "Set center" → recenter that pane
            if mode.value == "center" or ev.button == 3:
                _set_zoom_center(key, ev.xdata, ev.ydata, Hh, Ww)
                return
            # Middle click or Pan mode → start drag
            if mode.value == "pan" or ev.button == 2:
                _dragging = True; _drag_key = key; _drag_start = (ev.xdata, ev.ydata)
                return
            # Add STOP/PASS only when clicking FFT pane and left-click
            if key == 'fft' and ev.button in (1, None):
                kx, ky = imgcoord_to_normfreq(ev.xdata, ev.ydata, Ww, Hh)
                rpx = int(radius_px.value)
                if mode.value == "stop":
                    stops.append((ky, kx, rpx)); action_hist.append("stop")
                    with log_out:
                        print(f"STOP @ (ky={ky:+.4f}, kx={kx:+.4f}), r(px)={rpx}, depth={depth.value}, feather={feather_px.value}")
                elif mode.value == "pass":
                    passes.append((ky, kx, rpx)); action_hist.append("pass")
                    with log_out:
                        print(f"PASS @ (ky={ky:+.4f}, kx={kx:+.4f}), r(px)={rpx}, feather={feather_px.value}")
                update_preview()

        def on_release(ev):
            global _dragging, _drag_key
            if not _dragging or ev.inaxes not in ax2key or ev.xdata is None or ev.ydata is None:
                _dragging = False; _drag_key = None; return
            key = ax2key[ev.inaxes]
            if key != _drag_key:
                _dragging = False; _drag_key = None; return
            dx = ev.xdata - _drag_start[0]
            dy = ev.ydata - _drag_start[1]
            cx, cy = _zoom_centers[key]
            _zoom_centers[key] = (cx - dx, cy - dy)
            _dragging = False; _drag_key = None
            update_preview()

        _click_cids.append(fig.canvas.mpl_connect('button_press_event', on_press))
        _click_cids.append(fig.canvas.mpl_connect('button_release_event', on_release))

        plt.show()

def _set_zoom_center(key, xpix, ypix, H, W):
    _zoom_centers[key] = (float(np.clip(xpix, 0, W-1)), float(np.clip(ypix, 0, H-1)))
    update_preview()

def undo(_):
    if not action_hist:
        return
    last = action_hist.pop()
    if last == "stop" and stops:
        ky, kx, r = stops.pop()
        with log_out:
            print(f"Undo STOP (ky={ky:+.4f}, kx={kx:+.4f}, r(px)={r})")
    elif last == "pass" and passes:
        ky, kx, r = passes.pop()
        with log_out:
            print(f"Undo PASS (ky={ky:+.4f}, kx={kx:+.4f}, r(px)={r})")
    update_preview()

def clear_all(_):
    stops.clear(); passes.clear(); action_hist.clear()
    with log_out:
        print("Cleared all stops & passes")
    update_preview()

def reset_view(_):
    for k in _zoom_centers.keys():
        _zoom_centers[k] = None
    zoom.value = 1.0
    update_preview()

def save(_):
    data = {"stops":[{"ky":ky,"kx":kx,"r_px":r} for (ky,kx,r) in stops],
            "passes":[{"ky":ky,"kx":kx,"r_px":r} for (ky,kx,r) in passes],
            "params":{"feather_px": int(feather_px.value), "depth": float(depth.value), "mirror": bool(mirror.value)}}
    json.dump(data, open(preset.value,'w'), indent=2)
    with log_out:
        print("Saved:", preset.value)

def load_(_):
    path = preset.value
    if not os.path.exists(path):
        with log_out: print("Not found:", path); return
    data = json.load(open(path,'r'))
    stops.clear(); passes.clear(); action_hist.clear()
    for d in data.get("stops", []):  stops.append((float(d["ky"]), float(d["kx"]), int(d["r_px"])))
    for d in data.get("passes", []): passes.append((float(d["ky"]), float(d["kx"]), int(d["r_px"])))
    pf = data.get("params", {})
    if "feather_px" in pf: feather_px.value = int(pf["feather_px"])
    if "depth" in pf:      depth.value      = float(pf["depth"])
    if "mirror" in pf:     mirror.value     = bool(pf["mirror"])
    with log_out:
        print(f"Loaded: {path} (stops={len(stops)}, passes={len(passes)})")
    update_preview()

# ---- Reactivity & layout ----
for w in (z_slider, radius_px, feather_px, depth, mask_alpha, zoom, mode, mirror):
    w.observe(update_preview, names='value')
btn_undo.on_click(undo); btn_clear.on_click(clear_all); btn_reset.on_click(reset_view)
btn_save.on_click(save); btn_load.on_click(load_)

ui = W.VBox([
    z_slider,
    W.HBox([radius_px, feather_px, depth, mask_alpha]),
    W.HBox([zoom, mode, mirror, btn_reset]),
    W.HBox([btn_undo, btn_clear]),
    W.HBox([preset, btn_save, btn_load]),
    W.HTML("Left-click in <b>FFT</b>: add <b>STOP</b>/<b>PASS</b> disk. "
           "Middle-click or <i>Pan</i>: drag to pan the clicked pane. "
           "Right-click or <i>Set center</i>: recenter that pane. "
           "<b>radius</b>/<b>feather</b> are in <b>FFT pixels</b>. <b>depth</b> controls suppression strength.")
])
display(ui, plot_out, log_out)
update_preview()

Output()

Output()

# Cell 3 — apply to current slice and save (sanity check)

In [4]:
# Save the currently previewed slice after filtering
OUT_DIR = f"{DATA_DIR.rstrip('/')}/_fft_mask2d_preview"
os.makedirs(OUT_DIR, exist_ok=True)

z = z_slider.value
s_u = read_gray(files[z])
s, meta = to_float01(s_u)

# Build total mask (unshifted) using current STOP/PASS, depth, feather, mirror
_, _, H = build_masks(s.shape)

F  = fft2(s)
Sf = ifft2(F * H).real
Sf = (Sf.get() if hasattr(Sf, "get") else np.asarray(Sf)).astype(np.float32)

out_path = os.path.join(OUT_DIR, f"slice_{z:04d}.tif")
iio.imwrite(out_path, from_float01(Sf, meta))
print("Saved:", out_path)

def _to_np(x): return x.get() if hasattr(x, "get") else np.asarray(x)

Saved: /home/askiran/data/Peri_1/_fft_mask2d_preview/slice_0004.tif


# Cell 4 — apply to entire stack (per-slice) and save

In [5]:
# Apply current mask to all slices and save the full stack
OUT_DIR = f"{DATA_DIR.rstrip('/')}/_fft_mask2d_full"
os.makedirs(OUT_DIR, exist_ok=True)

# Build mask once for (Y,X)
Y, X = to_float01(read_gray(files[0]))[0].shape
_, _, H = build_masks((Y, X))  # total pass mask (unshifted)

# (Optional) also save masks (shifted) for inspection
def _save_mask(name, A):
    A16 = (np.clip((A.get() if hasattr(A,"get") else A), 0, 1)*65535 + 0.5).astype(np.uint16)
    iio.imwrite(os.path.join(OUT_DIR, name), A16)
_save_mask("mask_total_shifted.tif", cp.fft.fftshift(H))

try:
    from tqdm import tqdm
    it = tqdm(enumerate(files), total=len(files), desc="Filtering")
except Exception:
    it = enumerate(files)

for i, p in it:
    s_u = read_gray(p)
    s, meta = to_float01(s_u)
    F  = fft2(s)
    Sf = ifft2(F * H).real
    Sf = (Sf.get() if hasattr(Sf, "get") else np.asarray(Sf)).astype(np.float32)
    iio.imwrite(os.path.join(OUT_DIR, f"slice_{i:04d}.tif"), from_float01(Sf, meta))

print("Saved full stack to:", OUT_DIR)

Filtering: 100%|██████████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00, 69.60it/s]

Saved full stack to: /home/askiran/data/Small/_fft_mask2d_full


# === Convert .dm4 → per-slice TIFF + save/display pixel-size metadata =================
# Prefers HyperSpy (now with numba), falls back to NCEMpy.
# - Writes one TIFF per slice into DATA_DIR/_converted_tif
# - Saves a JSON sidecar per source file and a CSV summary with pixel sizes
# - If tifffile is present, writes OME-TIFF with PhysicalSizeX/Y in micrometers

In [5]:

DATA_DIR = "/home/askiran/data/Small" 

import os, json, warnings
import numpy as np
import imageio.v3 as iio
from glob import glob

# --- Set your input folder (where .dm4/.emd live). If DATA_DIR already set earlier, we keep it.
try:
    DATA_DIR
except NameError:
    DATA_DIR = os.getcwd()  # <-- change this to your dataset folder if you like

OUT_DIR        = f"{DATA_DIR.rstrip('/')}/_converted_tif"
EXPORT_PNG     = False         # also write PNG previews (8-bit, contrast-stretched)
EXPORT_JPEG    = False         # also write JPEG previews (8-bit, contrast-stretched)
CONTRAST_PCT   = (1, 99)
WRITE_OME_TIFF = True          # requires tifffile; will be disabled if not available

os.makedirs(OUT_DIR, exist_ok=True)

# --- Backends ----------------------------------------------------------------
warnings.filterwarnings("ignore", category=UserWarning)
import hyperspy.api as hs                # preferred
from ncempy.io import dm as _dm         # fallback
from ncempy.io import emd as _emd

try:
    import tifffile as tiff
    _HAVE_TIFFFILE = True
except Exception:
    _HAVE_TIFFFILE = False
    WRITE_OME_TIFF = False

# --- Helpers -----------------------------------------------------------------
def _to_uint8_vis(img, pct=(1,99)):
    a = np.asarray(img, np.float32)
    lo, hi = np.percentile(a, pct)
    if not np.isfinite(lo): lo = float(np.nanmin(a))
    if not np.isfinite(hi): hi = float(np.nanmax(a))
    if hi <= lo: hi = lo + 1e-6
    return (np.clip((a - lo)/(hi - lo), 0, 1)*255.0 + 0.5).astype(np.uint8)

def _flatten_to_stack(arr):
    """Return array as (Z, Y, X)."""
    arr = np.asarray(arr)
    if arr.ndim == 2:
        return arr[None, ...]
    Y, X = arr.shape[-2], arr.shape[-1]
    Z = int(np.prod(arr.shape[:-2]))
    return arr.reshape(Z, Y, X)

def _to_nm(val, unit):
    """Convert (value, unit) to nm if unit recognized."""
    if val is None or unit is None: return None
    try: v = float(val)
    except Exception: return None
    u = str(unit).strip().lower()
    if u in ("nm","nanometer","nanometre"): return v
    if u in ("ang","ångström","angström","angstrom","a","å"): return v * 0.1
    if u in ("µm","um","micrometer","micrometre"): return v * 1000.0
    if u in ("pm","picometer","picometre"): return v * 1e-3
    if u == "m":  return v * 1e9
    if u == "mm": return v * 1e6
    if u == "cm": return v * 1e7
    return None

def _extract_hyperspy_pixel_size(signal):
    """Return (py, uy, px, ux, py_nm, px_nm) from HyperSpy signal axes."""
    try:
        aY, aX = signal.axes_manager.signal_axes  # typically [Y, X]
    except Exception:
        try:
            aY = signal.axes_manager[-2]; aX = signal.axes_manager[-1]
        except Exception:
            return None, None, None, None, None, None
    try:
        py, uy = float(aY.scale), (aY.units or "")
        px, ux = float(aX.scale), (aX.units or "")
    except Exception:
        return None, None, None, None, None, None
    return py, uy, px, ux, _to_nm(py, uy), _to_nm(px, ux)

def _deep_find_scales(md):
    """Best-effort scan of NCEMpy metadata dict for pixel scales/units."""
    found = {}
    def walk(k, v):
        k_l = str(k).lower()
        if isinstance(v, (int,float,np.floating)):
            if any(x in k_l for x in ("xscale","x_scale","pixel size x","pixel_size_x","dx")):
                found.setdefault("px_x", float(v))
            if any(x in k_l for x in ("yscale","y_scale","pixel size y","pixel_size_y","dy")):
                found.setdefault("px_y", float(v))
            if k_l.endswith("scale"):
                found.setdefault("px_x", float(v))
                found.setdefault("px_y", float(v))
        if isinstance(v, (str, bytes)):
            vv = v.decode() if isinstance(v, bytes) else v
            if any(x in k_l for x in ("xunit","x_unit","unit x","units x")):
                found.setdefault("unit_x", vv)
            if any(x in k_l for x in ("yunit","y_unit","unit y","units y")):
                found.setdefault("unit_y", vv)
            if k_l.endswith("units"):
                found.setdefault("unit_x", vv)
                found.setdefault("unit_y", vv)
    def rec(d):
        if isinstance(d, dict):
            for kk, vv in d.items():
                walk(kk, vv); rec(vv)
        elif isinstance(d, (list, tuple)):
            for vv in d: rec(vv)
    try: rec(md)
    except Exception: pass
    return found

def read_with_hyperspy(path):
    s = hs.load(path, stack=True)
    # hs.load may return a list; choose the largest dataset
    if isinstance(s, (list, tuple)):
        s = max(s, key=lambda si: np.prod(np.shape(getattr(si, "data", np.array([])))))
    arr = np.asarray(s.data)
    stack = _flatten_to_stack(arr)
    py, uy, px, ux, py_nm, px_nm = _extract_hyperspy_pixel_size(s)
    meta = dict(backend="hyperspy", dtype=str(arr.dtype), shape=tuple(arr.shape),
                px_y=py, unit_y=uy, px_x=px, unit_x=ux, px_y_nm=py_nm, px_x_nm=px_nm)
    try:
        meta["hyperspy_metadata"] = s.metadata.as_dictionary()
    except Exception:
        pass
    return stack, meta

def read_with_ncempy(path):
    ext = os.path.splitext(path)[1].lower()
    if ext in (".dm3",".dm4"):
        out = _dm.read(path)
        arr = np.asarray(out.get("data"))
        md  = out.get("metadata", {})
    else:  # .emd
        out = _emd.read(path)
        # collect largest ndarray payload
        cand = []
        def _collect(v):
            if isinstance(v, np.ndarray): cand.append(v)
            elif isinstance(v, dict):
                for vv in v.values(): _collect(vv)
            elif isinstance(v, (list,tuple)):
                for vv in v: _collect(vv)
        _collect(out)
        if not cand:
            raise RuntimeError("No array payload found in EMD.")
        arr = np.asarray(max(cand, key=lambda a: a.size))
        md  = out
    stack = _flatten_to_stack(arr)
    info  = _deep_find_scales(md)
    px, ux = info.get("px_x"), info.get("unit_x")
    py, uy = info.get("px_y"), info.get("unit_y")
    meta = dict(backend="ncempy", dtype=str(arr.dtype), shape=tuple(arr.shape),
                px_y=py, unit_y=uy, px_x=px, unit_x=ux,
                px_y_nm=_to_nm(py, uy), px_x_nm=_to_nm(px, ux),
                ncempy_keys=list(md.keys()) if isinstance(md, dict) else None)
    return stack, meta

def read_dm_like(path):
    """Try HyperSpy first (better metadata), then NCEMpy."""
    try:
        return read_with_hyperspy(path)
    except Exception as e_hs:
        try:
            return read_with_ncempy(path)
        except Exception as e_nc:
            raise RuntimeError(f"{os.path.basename(path)}: HS failed ({e_hs}); NCEMpy failed ({e_nc})")

def write_slice(path_tif, img, meta):
    """
    Write a slice to disk.
    If tifffile available and PhysicalSize known, write an OME-TIFF with PhysicalSizeX/Y (µm).
    Otherwise, write a plain TIFF via tifffile or imageio.
    """
    if _HAVE_TIFFFILE:
        try:
            px_x_um = px_y_um = None
            if meta.get("px_x_nm") is not None: px_x_um = float(meta["px_x_nm"]) / 1000.0
            if meta.get("px_y_nm") is not None: px_y_um = float(meta["px_y_nm"]) / 1000.0
            if WRITE_OME_TIFF and (px_x_um is not None) and (px_y_um is not None):
                tiff.imwrite(
                    path_tif,
                    img,
                    photometric="minisblack",
                    ome=True,
                    metadata={
                        "axes": "YX",
                        "PhysicalSizeX": px_x_um, "PhysicalSizeXUnit": "micrometer",
                        "PhysicalSizeY": px_y_um, "PhysicalSizeYUnit": "micrometer",
                    },
                )
            else:
                tiff.imwrite(path_tif, img, photometric="minisblack")
            return
        except Exception:
            pass  # robust fallback
    iio.imwrite(path_tif, img)

# --- Find inputs (.dm4/.emd) -------------------------------------------------
inputs = []
for pat in ("*.dm4","*.emd"):   # add "*.dm3" if you also want DM3
    inputs += glob(os.path.join(DATA_DIR, pat))
inputs.sort()
print(f"[convert] Found {len(inputs)} inputs in {DATA_DIR}")

# --- Convert & log -----------------------------------------------------------
rows = []
if inputs:
    try:
        from tqdm import tqdm
        iterator = tqdm(inputs, desc="Converting")
    except Exception:
        iterator = inputs

    for path in iterator:
        name = os.path.basename(path)
        try:
            stack, meta = read_dm_like(path)
        except Exception as e:
            print("  SKIP:", name, "→", e)
            continue

        Z, Y, X = stack.shape
        # Sidecar JSON (per source)
        json_path = os.path.join(OUT_DIR, f"{os.path.splitext(name)[0]}_meta.json")
        with open(json_path, "w") as jf:
            json.dump(meta, jf, indent=2)

        # Write slices
        stem = os.path.splitext(name)[0]
        for zi in range(Z):
            tif_path = os.path.join(OUT_DIR, f"{stem}_z{zi:04d}.tif")
            write_slice(tif_path, stack[zi], meta)
            if EXPORT_PNG:
                iio.imwrite(os.path.join(OUT_DIR, f"{stem}_z{zi:04d}.png"), _to_uint8_vis(stack[zi], CONTRAST_PCT))
            if EXPORT_JPEG:
                iio.imwrite(os.path.join(OUT_DIR, f"{stem}_z{zi:04d}.jpg"), _to_uint8_vis(stack[zi], CONTRAST_PCT))

        rows.append(dict(
            file=name, Z=Z, Y=Y, X=X, dtype=str(stack.dtype),
            px_y=meta.get("px_y"), unit_y=meta.get("unit_y"),
            px_x=meta.get("px_x"), unit_x=meta.get("unit_x"),
            px_y_nm=meta.get("px_y_nm"), px_x_nm=meta.get("px_x_nm"),
            backend=meta.get("backend")
        ))

# --- Display summary + save CSV ---------------------------------------------
def _fmt(v):
    if v is None: return "—"
    if isinstance(v, (int,float,np.floating)): return f"{v:.6g}"
    return str(v)

if rows:
    print(f"[done] Wrote {sum(r['Z'] for r in rows)} TIFF slice(s) to:\n  {OUT_DIR}\n")
    print("Metadata summary (first 10):")
    for r in rows[:10]:
        print(
            f"{r['file']}\n"
            f"  shape: (Z={r['Z']}, Y={r['Y']}, X={r['X']}), dtype={r['dtype']}, backend={r['backend']}\n"
            f"  px_y: {_fmt(r['px_y'])} {r['unit_y'] or ''}  (~ {_fmt(r['px_y_nm'])} nm)\n"
            f"  px_x: {_fmt(r['px_x'])} {r['unit_x'] or ''}  (~ {_fmt(r['px_x_nm'])} nm)"
        )
    if len(rows) > 10:
        print(f"... ({len(rows)-10} more files)")

    csv_path = os.path.join(OUT_DIR, "_meta_summary.csv")
    header = ["file","Z","Y","X","dtype","px_y","unit_y","px_x","unit_x","px_y_nm","px_x_nm","backend"]
    with open(csv_path, "w") as f:
        f.write(",".join(header) + "\n")
        for r in rows:
            f.write(",".join(str(r.get(k,"")) for k in header) + "\n")
    print("\nSaved CSV:", csv_path)
else:
    print("No .dm4/.emd files converted.")

print("\nTip: For your FFT UI, you can now set:  DATA_DIR = OUT_DIR")


[convert] Found 2 inputs in /home/askiran/data/Small


Converting:   0%|                                                                                 | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

Converting:  50%|████████████████████████████████████▌                                    | 1/2 [00:00<00:00,  5.18it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

Converting: 100%|█████████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00,  6.26it/s]

[done] Wrote 2 TIFF slice(s) to:
  /home/askiran/data/Small/_converted_tif

Metadata summary (first 10):
840X_BM-OneView_200kV_0016.dm4
  shape: (Z=1, Y=4096, X=4096), dtype=float32, backend=hyperspy
  px_y: 0.00715701 1/nm  (~ — nm)
  px_x: 0.00715701 1/nm  (~ — nm)
Pd48 STEM HAADF-DF4-DF2-BF 3.00 Mx 0033.emd
  shape: (Z=1, Y=1024, X=1024), dtype=float64, backend=hyperspy
  px_y: 0.0324929 nm  (~ 0.0324929 nm)
  px_x: 0.0324929 nm  (~ 0.0324929 nm)

Saved CSV: /home/askiran/data/Small/_converted_tif/_meta_summary.csv

Tip: For your FFT UI, you can now set:  DATA_DIR = OUT_DIR


# === Robust converter: .dm4 + all 2D datasets in .emd → per-slice TIFFs + metadata =========
# - De-duplicates identical EMD datasets via content fingerprint
# - One subfolder per dataset: OUT_DIR/<stem>__<dataset_name>/slice_0000.tif
# - Saves per-dataset JSON and a CSV summary
# Requires: hyperspy, ncempy; optional: tifffile

In [9]:
# === Convert .dm4 + ALL 2D datasets in .emd → per-slice TIFFs + metadata =================
DATA_DIR = "/home/askiran/data/Small" 

import os, re, json, warnings
import numpy as np
import imageio.v3 as iio
from glob import glob

# -------- CONFIG --------
try:
    DATA_DIR
except NameError:
    DATA_DIR = os.getcwd()       # <-- change input folder if needed

OUT_DIR         = f"{DATA_DIR.rstrip('/')}/_converted_tif"
EXPORT_PNG      = False
EXPORT_JPEG     = False
CONTRAST_PCT    = (1, 99)
WRITE_OME_TIFF  = True          # needs tifffile; disabled automatically if missing
# Skip datasets with these titles (optional):
EMD_SKIP_TITLES = {"Preview", "Thumbnail"}  # add "Small" here if you know it's redundant in your files
# Dedup granularity: if True, drop datasets that have identical shape/dtype & (mean,std,min,max)
DEDUP_BY_STATS  = True

os.makedirs(OUT_DIR, exist_ok=True)
warnings.filterwarnings("ignore", category=UserWarning)

# ---- backends
import hyperspy.api as hs
from ncempy.io import dm as _dm
from ncempy.io import emd as _emd

try:
    import tifffile as tiff
    _HAVE_TIFFFILE = True
except Exception:
    _HAVE_TIFFFILE = False
    WRITE_OME_TIFF = False

# -------- helpers --------
def _safe(s):
    s = re.sub(r"[^\w\-.]+", "_", str(s))
    return s[:80] or "ds"

def _to_uint8_vis(img, pct=(1,99)):
    a = np.asarray(img, np.float32)
    lo, hi = np.percentile(a, pct)
    if not np.isfinite(lo): lo = float(np.nanmin(a))
    if not np.isfinite(hi): hi = float(np.nanmax(a))
    if hi <= lo: hi = lo + 1e-6
    return (np.clip((a - lo)/(hi - lo), 0, 1)*255.0 + 0.5).astype(np.uint8)

def _flatten_to_stack(arr):
    """Return array as (Z, Y, X). Last two dims are Y,X; all preceding flattened to Z."""
    arr = np.asarray(arr)
    if arr.ndim < 2:
        raise ValueError(f"Need at least 2D array for images, got shape {arr.shape}")
    Y, X = arr.shape[-2], arr.shape[-1]
    Z = int(np.prod(arr.shape[:-2])) if arr.ndim > 2 else 1
    return arr.reshape(Z, Y, X)

def _to_nm(val, unit):
    if val is None or unit is None: return None
    try: v = float(val)
    except Exception: return None
    u = str(unit).strip().lower()
    if u in ("nm","nanometer","nanometre"): return v
    if u in ("ang","ångström","angström","angstrom","a","å"): return v * 0.1
    if u in ("µm","um","micrometer","micrometre"): return v * 1000.0
    if u in ("pm","picometer","picometre"): return v * 1e-3
    if u == "m":  return v * 1e9
    if u == "mm": return v * 1e6
    if u == "cm": return v * 1e7
    return None

def _extract_hs_px(signal):
    """Return (py, uy, px, ux, py_nm, px_nm) if 2D signal; else (None,...)."""
    try:
        if signal.axes_manager.signal_dimension != 2:
            return (None,)*6
        aY, aX = signal.axes_manager.signal_axes
        py, uy = float(aY.scale), (aY.units or "")
        px, ux = float(aX.scale), (aX.units or "")
        return py, uy, px, ux, _to_nm(py, uy), _to_nm(px, ux)
    except Exception:
        return (None,)*6

def write_slice(path_tif, img, meta):
    """OME-TIFF with PhysicalSizeX/Y (µm) if possible; otherwise plain TIFF."""
    if _HAVE_TIFFFILE:
        try:
            px_x_um = px_y_um = None
            if meta.get("px_x_nm") is not None: px_x_um = float(meta["px_x_nm"]) / 1000.0
            if meta.get("px_y_nm") is not None: px_y_um = float(meta["px_y_nm"]) / 1000.0
            if WRITE_OME_TIFF and (px_x_um is not None) and (px_y_um is not None):
                tiff.imwrite(
                    path_tif, img, photometric="minisblack", ome=True,
                    metadata={
                        "axes": "YX",
                        "PhysicalSizeX": px_x_um, "PhysicalSizeXUnit": "micrometer",
                        "PhysicalSizeY": px_y_um, "PhysicalSizeYUnit": "micrometer",
                    },
                )
            else:
                tiff.imwrite(path_tif, img, photometric="minisblack")
            return
        except Exception:
            pass
    iio.imwrite(path_tif, img)

def _stats_fingerprint(stack):
    """Lightweight dedup fingerprint: (shape, dtype, mean, std, min, max) rounded."""
    a = np.asarray(stack, dtype=np.float64)  # safe stats
    m  = float(np.nanmean(a))
    sd = float(np.nanstd(a))
    mn = float(np.nanmin(a))
    mx = float(np.nanmax(a))
    return (tuple(stack.shape), str(stack.dtype),
            round(m, 6), round(sd, 6), round(mn, 6), round(mx, 6))

# -------- readers --------
def read_dm4_single(path):
    out = _dm.read(path)
    arr = np.asarray(out.get("data"))
    md  = out.get("metadata", {})
    # pixel size: best-effort keys within md vary; leave None unless you know exact keys
    meta = dict(backend="ncempy.dm", dtype=str(arr.dtype), shape=tuple(arr.shape),
                px_y=None, unit_y=None, px_x=None, unit_x=None,
                px_y_nm=None, px_x_nm=None, raw_keys=list(md.keys()) if isinstance(md, dict) else None)
    stack = _flatten_to_stack(arr)
    return [("image", stack, meta)]

def read_emd_all_datasets_hyperspy(path):
    """Enumerate all 2D signals via HyperSpy (no stacking); dedup identical content."""
    sigs = hs.load(path, stack=False, lazy=False)
    if not isinstance(sigs, (list, tuple)):
        sigs = [sigs]
    out = []
    seen = set()
    for i, s in enumerate(sigs):
        if getattr(s.axes_manager, "signal_dimension", 0) != 2:
            continue
        # optional skip by title
        title = None
        try: title = s.metadata.General.title
        except Exception: pass
        if title and title in EMD_SKIP_TITLES:
            continue

        arr = np.asarray(s.data)
        if arr.size == 0:
            continue
        stack = _flatten_to_stack(arr)  # (Z,Y,X)
        # dedup
        fp = _stats_fingerprint(stack) if DEDUP_BY_STATS else (id(arr),)
        if fp in seen:
            continue
        seen.add(fp)

        # name
        name = None
        if title: name = title
        if not name:
            name = getattr(s, "name", None)
        # add detector or path hint if available
        hint = None
        for pathkey in ("Original_filename", "record_by", "path"):
            try:
                hint = hint or str(getattr(s.metadata, pathkey))
            except Exception:
                pass
        ds_name = _safe(name or f"sig{i}")

        py, uy, px, ux, py_nm, px_nm = _extract_hs_px(s)
        meta = dict(backend="hyperspy", dtype=str(arr.dtype), shape=tuple(arr.shape),
                    px_y=py, unit_y=uy, px_x=px, unit_x=ux, px_y_nm=py_nm, px_x_nm=px_nm,
                    title=title, hint=hint)
        out.append((ds_name, stack, meta))
    return out

def read_emd_all_datasets_ncempy(path):
    """Fallback traversal for .emd using NCEMpy; de-dup via stats."""
    raw = _emd.read(path)
    datasets = []
    def visit(node, keypath="root"):
        if isinstance(node, np.ndarray):
            if node.ndim >= 2 and node.shape[-2] > 1 and node.shape[-1] > 1:
                datasets.append((keypath, np.asarray(node)))
        elif isinstance(node, dict):
            for k, v in node.items():
                visit(v, f"{keypath}/{k}")
        elif isinstance(node, (list, tuple)):
            for idx, v in enumerate(node):
                visit(v, f"{keypath}[{idx}]")
    visit(raw)

    out, seen = [], set()
    for j, (kp, arr) in enumerate(datasets):
        try:
            stack = _flatten_to_stack(arr)
        except Exception:
            continue
        fp = _stats_fingerprint(stack) if DEDUP_BY_STATS else (id(arr),)
        if fp in seen:
            continue
        seen.add(fp)
        ds_name = _safe(os.path.basename(kp) or f"emd{j}")
        meta = dict(backend="ncempy.emd", dtype=str(arr.dtype), shape=tuple(arr.shape),
                    px_y=None, unit_y=None, px_x=None, unit_x=None,
                    px_y_nm=None, px_x_nm=None, path=kp)
        out.append((ds_name, stack, meta))
    return out

def read_emd_all_datasets(path):
    try:
        ds = read_emd_all_datasets_hyperspy(path)
        if ds:
            return ds
    except Exception:
        pass
    return read_emd_all_datasets_ncempy(path)

# -------- scan inputs --------
inputs = glob(os.path.join(DATA_DIR, "*.dm4")) + glob(os.path.join(DATA_DIR, "*.emd"))
inputs.sort()
print(f"[convert] Found {len(inputs)} inputs in {DATA_DIR}")

# -------- convert --------
rows = []
if inputs:
    try:
        from tqdm import tqdm
        iterator = tqdm(inputs, desc="Converting")
    except Exception:
        iterator = inputs

    for path in iterator:
        stem = os.path.splitext(os.path.basename(path))[0]
        ext  = os.path.splitext(path)[1].lower()

        datasets = read_dm4_single(path) if ext == ".dm4" else read_emd_all_datasets(path)
        if not datasets:
            print("  SKIP (no 2D datasets):", os.path.basename(path))
            continue

        for ds_name, stack, meta in datasets:
            Z, Y, X = stack.shape
            subdir = os.path.join(OUT_DIR, f"{stem}__{ds_name}")
            os.makedirs(subdir, exist_ok=True)

            # sidecar metadata per dataset
            with open(os.path.join(subdir, "_meta.json"), "w") as jf:
                json.dump(meta, jf, indent=2)

            # write slices
            for zi in range(Z):
                tif_path = os.path.join(subdir, f"slice_{zi:04d}.tif")
                write_slice(tif_path, stack[zi], meta)
                if EXPORT_PNG:
                    iio.imwrite(os.path.join(subdir, f"slice_{zi:04d}.png"), _to_uint8_vis(stack[zi], CONTRAST_PCT))
                if EXPORT_JPEG:
                    iio.imwrite(os.path.join(subdir, f"slice_{zi:04d}.jpg"), _to_uint8_vis(stack[zi], CONTRAST_PCT))

            rows.append(dict(
                file=os.path.basename(path),
                dataset=ds_name, Z=Z, Y=Y, X=X, dtype=str(stack.dtype),
                px_y=meta.get("px_y"), unit_y=meta.get("unit_y"),
                px_x=meta.get("px_x"), unit_x=meta.get("unit_x"),
                px_y_nm=meta.get("px_y_nm"), px_x_nm=meta.get("px_x_nm"),
                backend=meta.get("backend")
            ))

# -------- report & CSV --------
def _fmt(v):
    if v is None: return "—"
    if isinstance(v, (int,float,np.floating)): return f"{v:.6g}"
    return str(v)

if rows:
    total_slices = sum(r["Z"] for r in rows)
    print(f"[done] Wrote {total_slices} TIFF slice(s) into subfolders under:\n  {OUT_DIR}\n")
    print("Datasets (first 10):")
    for r in rows[:10]:
        print(
            f"{r['file']}  →  {r['dataset']}\n"
            f"  shape: (Z={r['Z']}, Y={r['Y']}, X={r['X']}), dtype={r['dtype']}, backend={r['backend']}\n"
            f"  px_y: {_fmt(r['px_y'])} {r['unit_y'] or ''}  (~ {_fmt(r['px_y_nm'])} nm)\n"
            f"  px_x: {_fmt(r['px_x'])} {r['unit_x'] or ''}  (~ {_fmt(r['px_x_nm'])} nm)\n"
        )
    if len(rows) > 10:
        print(f"... ({len(rows)-10} more datasets)")

    csv_path = os.path.join(OUT_DIR, "_meta_summary.csv")
    header = ["file","dataset","Z","Y","X","dtype","px_y","unit_y","px_x","unit_x","px_y_nm","px_x_nm","backend"]
    with open(csv_path, "w") as f:
        f.write(",".join(header) + "\n")
        for r in rows:
            f.write(",".join(str(r.get(k,"")) for k in header) + "\n")
    print("Saved CSV:", csv_path)
else:
    print("No convertible datasets found.")

print("\nTip: For your FFT UI, point to a dataset folder, e.g.:")
print("  DATA_DIR = os.path.join(OUT_DIR, '<file_stem>__<dataset_name>')")

[convert] Found 1 inputs in /home/askiran/data/Small


Converting: 100%|█████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 11.40it/s]

[done] Wrote 4 TIFF slice(s) into subfolders under:
  /home/askiran/data/Small/_converted_tif

Datasets (first 10):
Pd48 STEM HAADF-DF4-DF2-BF 3.00 Mx 0033.emd  →  DF2
  shape: (Z=1, Y=1024, X=1024), dtype=float64, backend=hyperspy
  px_y: 0.0324929 nm  (~ 0.0324929 nm)
  px_x: 0.0324929 nm  (~ 0.0324929 nm)

Pd48 STEM HAADF-DF4-DF2-BF 3.00 Mx 0033.emd  →  DF4
  shape: (Z=1, Y=1024, X=1024), dtype=float64, backend=hyperspy
  px_y: 0.0324929 nm  (~ 0.0324929 nm)
  px_x: 0.0324929 nm  (~ 0.0324929 nm)

Pd48 STEM HAADF-DF4-DF2-BF 3.00 Mx 0033.emd  →  HAADF
  shape: (Z=1, Y=1024, X=1024), dtype=float64, backend=hyperspy
  px_y: 0.0324929 nm  (~ 0.0324929 nm)
  px_x: 0.0324929 nm  (~ 0.0324929 nm)

Pd48 STEM HAADF-DF4-DF2-BF 3.00 Mx 0033.emd  →  BF
  shape: (Z=1, Y=1024, X=1024), dtype=float64, backend=hyperspy
  px_y: 0.0324929 nm  (~ 0.0324929 nm)
  px_x: 0.0324929 nm  (~ 0.0324929 nm)

Saved CSV: /home/askiran/data/Small/_converted_tif/_meta_summary.csv

Tip: For your FFT UI, point to a 